## Pairwise grading

In order to assess the specificity of certain disclosures, we are going to perform pairwise grading. This means starting with the positively identified samples and then putting them up against one another in a grading schema (based on categories from Oppong-Tawiah and Webster 2023)

In [44]:
import sys
sys.path.append('..')
import utils
from path import Path
import openai
import os
from dotenv import load_dotenv
import pandas as pd

# Load environment variables from .env file
load_dotenv()

# Get OpenRouter API key from environment variables
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError("Please set OPENROUTER_API_KEY in your .env file or environment variables")


In [45]:
results_path = '../results/SAO_RAG_RESULTS'
results_path = Path(results_path)

In [46]:
results = utils.load_company_data(results_path)

In [47]:
results

,company,original_index,year,text,climate_litigation,retrieval_similarity,num_examples_used
0,RCL,25,2014,We believe that the impact of ships on the glo...,climate_litigation: 0,0.564889,5
1,RCL,48,2014,To the extent the tonnage tax laws of these co...,climate_litigation: 0,0.531344,5
2,RCL,40,2014,"dollar, including, among others, the British p...",climate_litigation: 0,0.528860,5
3,RCL,24,2014,"The ISM Code is mandatory for all vessels, inc...",climate_litigation: 0,0.523801,5
4,RCL,41,2014,An increase in fuel prices not only impacts ou...,climate_litigation: 0,0.516437,5
...,...,...,...,...,...,...,...
52505,GLP,64,2016,"For example, our partnership agreement: ·\nper...",climate_litigation: 0,0.484702,5
52506,GLP,88,2016,We operate our business under three segments: ...,climate_litigation: 0,0.484497,5
52507,GLP,104,2016,Our gross profit for 2014 was negatively impac...,climate_litigation: 0,0.482937,5
52508,GLP,285,2016,"The fees, which are based upon an estimate of ...",climate_litigation: 0,0.482671,5


Extracting the binary number from the climate litigation assignment 

In [48]:
trouble_indices = []

In [49]:
results['climate_litigation_binary'] = results.apply(
    lambda row: utils.extract_flag(row['climate_litigation'], row.name, trouble_indices), 
    axis=1
)

Subsetting to only the positively identified phrases

In [50]:
results_1 = results[results['climate_litigation_binary'] == 1]

In [51]:
len(results_1)

616

In [52]:
#calculate the number of pairwise comparisons we need to make; note this is mn(n−1) rather than mn(n−1)/2 because we always need to swap the presentation order of the responses si and sj
#reference: Gao 2025 

total_comparisons = len(results_1) * (len(results_1) - 1)
total_comparisons


378840

## Pairwise prompting

In [53]:
client = openai.OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
def get_pairwise_prompt(sentence_1, sentence_2):
    return (
        "You are comparing two sections to decide which is MORE SPECIFIC about CLIMATE CHANGE LITIGATION.\n"
        "\n"
        "Use the literature notion of specificity: level of detail and precision about place, time, numbers, or descriptive/sensory facts "
        "('spatio-temporal/descriptive' vs. generalized wording). Apply this ONLY to climate change litigation content.\n"
        "\n"
        "PROCESS\n"
        "1) Independently assess Section 1, then Section 2.\n"
        "2) For each section, judge specificity based on:\n"
        "   - PRECISION of anchors: named parties/case title/docket, forum or jurisdiction, statute/regulation/policy, procedural posture, dates/time frames, monetary/quantified remedies or thresholds.\n"
        "   - LINKAGE: whether those anchors are clearly tied to climate claims (not just generic legal or sustainability talk).\n"
        "   - FALSIFIABILITY: whether the details are concrete enough to be checked or challenged (e.g., exact court, filing date, cited rule).\n"
        "   - CONTEXTUAL CLARITY: whether the details fit together coherently (who did what, where, when, under which rule, seeking what outcome).\n"
        "Ignore specificity about non-climate topics.\n"
        "\n"
        "SCORING GUIDELINES (no counting—judge by presence, precision, and linkage strength):\n"
        "0: No climate litigation content at all.\n"
        "10: Climate issues mentioned but not litigation.\n"
        "20: Litigation is referenced but only in generic terms; no precise legal anchor or linkage to climate claims.\n"
        "40: At least one legal anchor appears (e.g., forum, party, statute) but is vague or only loosely tied to climate claims; timing/place/amounts are broad or implied.\n"
        "60: Clear, specific legal anchor(s) precisely linked to climate claims (e.g., named agency action, identifiable forum or statute, plausible timeframe or remedy), but some elements remain general or implicit.\n"
        "80: Multiple precise and well-linked anchors (e.g., named party or case title AND specific forum/jurisdiction AND concrete timeframe or remedy); coherent and checkable, minor gaps only.\n"
        "90–100: Highly specific and verifiable (e.g., case name or docket; exact court; specific date(s) or stage; cited rule/statute; defined remedy/amounts), tightly and explicitly tied to climate claims with strong internal coherence.\n"
        "Choose an integer score that best fits the description above.\n"
        "\n"
        "DECISION\n"
        "- After scoring both independently, choose the MORE SPECIFIC section.\n"
        "- If scores differ by ≥20, pick the higher score.\n"
        "- If |diff| ≤ 10, apply tie-breakers IN ORDER:\n"
        "  (i) Prefer the section that names a case, docket, or parties with a specific forum/jurisdiction.\n"
        "  (ii) Prefer the section with clearer procedural posture and explicit linkage to a climate rule/policy/statute.\n"
        "  (iii) Prefer the section with more precise spatio-temporal or monetary details that are checkable.\n"
        "  (iv) If still tied, output \"section 2\" (bias control).\n"
        "\n"
        "RULES\n"
        "- OUTPUT MUST BE VALID JSON ONLY (no extra text, no comments, no trailing commas).\n"
        "\n"
        "OUTPUT FORMAT (exact schema, do not provide any other text):\n"
        "[\n"
        "  {\n"
        "    \"binary result\": \"section 1\" or \"section 2\",\n"
        "    \"section_1\": <integer 0-100>,\n"
        "    \"section_2\": <integer 0-100>\n"
        "  }\n"
        "]\n"
        "\n"
        "SECTIONS\n"
        f"Section 1: {sentence_1}\n"
        f"Section 2: {sentence_2}\n"
    )

In [ ]:
sentence_1 = results_1.iloc[13]['text']
sentence_2 = results_1.iloc[16]['text']
message = get_pairwise_prompt(sentence_1, sentence_2)
from pprint import pprint
pprint(message)

('You are comparing two sections to decide which is MORE SPECIFIC about '
 'CLIMATE CHANGE LITIGATION.\n'
 '\n'
 'Use the literature notion of specificity: level of detail and precision '
 'about place, time, numbers, or descriptive/sensory facts '
 "('spatio-temporal/descriptive' vs. generalized wording). Apply this ONLY to "
 'climate change litigation content.\n'
 '\n'
 'PROCESS\n'
 '1) Independently assess Section 1, then Section 2.\n'
 '2) For each section, judge specificity based on:\n'
 '   - PRECISION of anchors: named parties/case title/docket, forum or '
 'jurisdiction, statute/regulation/policy, procedural posture, dates/time '
 'frames, monetary/quantified remedies or thresholds.\n'
 '   - LINKAGE: whether those anchors are clearly tied to climate claims (not '
 'just generic legal or sustainability talk).\n'
 '   - FALSIFIABILITY: whether the details are concrete enough to be checked '
 'or challenged (e.g., exact court, filing date, cited rule).\n'
 '   - CONTEXTUAL CLARI

In [ ]:
SYSTEM_MESSAGE = (
    "You are a legal and environmental disclosure expert. Previously, someone else extracted all of the sentences from several companies' annual reports that related to climate litigation. Now, we are interested in assessing the specificity of these disclosures.\n\n"
    "Specificity captures the level of detail and precision, measured by the number of specific details related to place, time, numbers, or the five senses (i.e., “descriptive” and “spatio-temporal” words), as opposed to “generalized words”.\n\n"
    "The most important part of your job is choosing which section is more specific. So focus on that first and foremost. You only respond in the JSON format, no other text. EVER."
)

In [ ]:
response = client.chat.completions.create(
    model="meta-llama/llama-3.1-70b-instruct",
    messages=[
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": message}
    ],
    temperature=0.0
)
print(response.choices[0].message.content)


[
  {
    "binary result": "section 1",
    "section_1": 60,
    "section_2": 40
  }
]


I'm having the problem that if I assess with the more powerful LLM it is more consistent but the one that proved most accurate in our other tasks is failing, always saying section 2 (mostly) and also struggling in consistency. Maybe it needs the guardrails of few shot learning to be effective? I wonder if I could justify a different model in the pairwise grading than in the original work. 

We can justify from Gao et al. 2025 that the best open source model for pairwise grading is meta-llama/llama-3.1-70b-instruct. The problem is just then showing why we did what we did in the first task. I think that's doable, we just need to cast the original problem as it was, a more straightforward classification task that should be achievable for all LLMs. 

## Running a small test against state of the art GPTo5

Think of this as our 'groundtruth' since it has been shown to be so successful on these types of tasks

## Concept-guided chain-of-thought

Start by including more than just the climate litigation sentences; we want to show that there are some that are not climate litigation at all. However, we will tell the LLM what it was graded before. 

In [13]:
results_0 = results[results['climate_litigation_binary'] == 0]
results_0 = results_0.sample(n=308, random_state=42)


In [17]:
all_results = pd.concat([results_1, results_0])

## Setting the prompts

In [192]:
SYSTEM_PROMPT = """
You have been given a text. Please look at the text and respond in relation to that text at each step. 

You are a legal and environmental disclosure expert. 
"""

In [193]:
def get_prompt_1(text):
    return f"Summarize the following text: {text}"

def get_prompt_2(text, summary):
    return (
        "Please assess whether the following text is climate litigation.\n\n"
        "Previously, someone else extracted all of the sentences from several companies' annual reports that related to climate litigation. "
        "Now, we are interested in assessing the specificity of these disclosures.\n\n"
        "Now, we are just seeing if you agree with them.\n"
        "Climate litigation refers to legal actions that materially concern climate change science, policy, or law. These include, but are not limited to:\n"
        "- Lawsuits targeting false or misleading climate claims (e.g. greenwashing)\n"
        "- Legal actions over a company’s contribution to climate-related impacts\n"
        "- Efforts to force climate alignment through human rights or fiduciary duty arguments\n"
        "- Failure to disclose climate-related risks or impacts\n"
        "- Breaches of climate-related regulations\n"
        "- Litigation seeking damages for harms caused by climate change\n"
        "- Legal challenges to regulatory approvals on the basis of climate misalignment\n\n"
        "You will be given a summary of the text, as well as the original text.\n"
        f"Summary: {summary}\n\nOriginal text: {text}"
    )

def get_prompt_3(text, summary, classification):
    return (
        "For your knowledge, the text was previously classified as follows:\n"
        f"Summary: {summary}\n\nOriginal text: {text}\n\nClassification: {classification}\n"
        "Please only respond with an acknowledgement of this information."
    )


In [194]:
_BREAKDOWN_JSON_SCHEMA = r"""
Return ONLY valid JSON with this exact schema and keys (no code fences, no extra text):

{
  "parties": [string],
  "case_title_or_docket": string or null,
  "forum_or_jurisdiction": string or null,
  "statute_reg_policy": [string],
  "procedural_posture": string or null,
  "dates_or_timeframes": [string],
  "monetary_or_quantified": [string],
  "climate_linkage_summary": string,
  "climate_linkage_quotes": [string],
  "checkable_elements": [string],
  "who": string or null,
  "what": string or null,
  "where": string or null,
  "when": string or null,
  "under_which_rule": string or null,
  "sought_outcome": string or null,
  "evidence_quotes": [string]
}

Rules:
- Only consider these categories in relation to climate litigation. If there are dates, parties, monetary amounts, etc. that are not related to climate litigation, ignore them.
- Populate every key. Use [] for unknown lists, and null for unknown single values.
- Prefer short verbatim snippets from the text for any *quotes* arrays.
- Do NOT invent details not supported by the text. Ignore specificity about non-climate topics.
"""

def get_prompt_4(text, summary):
    return (
        "Build a concept-specific breakdown of SPECIFICITY for climate litigation.\n"
        "Focus on anchors (parties/case/docket, forum/jurisdiction, statute/reg/policy, posture, dates, monetary amounts), "
        "climate linkage, falsifiability (checkable details), and contextual clarity (who/what/where/when/under which rule/seeking what).\n\n"
        "It is very impotant that you only fill in the following schema with details that are DIRECTLY RELATED to climate litigation. Many text chunks will include details about other climate related regulations, monetary incentives, etc. but ONLY IF the text provides the following details about CLIMATE LITIGATION ITSELF, should you fill them in. Otherwise, they should be given null or empty lists."
        f"Summary: {summary}\n\nOriginal text:\n{text}\n\n"
        + _BREAKDOWN_JSON_SCHEMA
    )

In [196]:
_SPEC_LABEL_OPTIONS = [
    "not climate litigation",
    "mention only",           # acknowledges CL exists, zero specifics
    "vague disclosure",        # generic statements, no verifiable details
    "basic disclosure",        # some details but missing most anchors
    "detailed disclosure",     # multiple anchors, partially verifiable
    "highly specific",         # comprehensive, checkable information
]

def get_prompt_5(text, summary, classification, breakdown_json):
    return (
        "Given the summary, original text, CL classification, and the structured breakdown, choose ONE specificity label.\n"
        "Base your choice on:\n"
        " - PRECISION of anchors (parties/case/docket, forum, statute/reg/policy, posture, dates, monetary/quantified).\n"
        " - LINKAGE to climate claims (not generic legal/ESG talk).\n"
        " - FALSIFIABILITY (checkable details: exact court, date, cited rule, quantified remedy, etc.).\n"
        " - CONTEXTUAL CLARITY (who did what, where, when, under which rule, seeking what outcome).\n"
        "Ignore specificity about non-climate and non-litigation topics\n\n"
        f"Return ONLY one of: {', '.join(_SPEC_LABEL_OPTIONS)}\n\n"
        f"Summary: {summary}\n"
        f"Original text: {text}\n"
        f"Previously classified as climate litigation? {classification}\n"
        f"Breakdown JSON: {breakdown_json}"
    )

In [ ]:
def get_prompt_6(full_message_1, full_message_2):
    return (
        "You will compare two climate litigation assessments for SPECIFICITY.\n\n"
        
        "TASK: Determine which assessment provides MORE SPECIFIC details about climate litigation.\n"
        "Consider: precision of anchors (parties/case/forum/statute/dates/amounts), "
        "climate linkage strength, falsifiability (checkable details), and contextual clarity.\n\n"
        
        "CRITICAL: Do not favor an assessment based on its position (Text 1 vs Text 2). "
        "Base your judgment solely on content quality.\n\n"
        
        "SPECIAL CASES:\n"
        "- If exactly one assessment identifies climate litigation content, choose that one.\n"
        "- If BOTH assessments find NO climate litigation content, return 'neither'.\n"
        "- If both find climate litigation but one is clearly more specific, choose the more specific one.\n"
        "- Only use 'neither' when BOTH texts contain zero climate litigation content.\n\n"
        
        "EVALUATION PROCESS:\n"
        "1. First, analyze Text 1 independently:\n"
        f"   {full_message_1}\n\n"
        
        "2. Second, analyze Text 2 independently:\n"
        f"   {full_message_2}\n\n"
        
        "3. Direct comparison: Which provides more specific, verifiable climate litigation details?\n\n"
        
        "FORMAT:\n"
        "Provide your reasoning, then end with exactly one of these on a new line:\n"
        "FINAL: text 1\n"
        "FINAL: text 2\n"
        "FINAL: neither"
    )

In [198]:
def get_prompt_7(results):
    return (
        f"In the following content, extract the final decision. "
        f"Return ONLY 'text 1' or 'text 2' or 'neither' (no other text). "
        f"If absent, infer from the last explicit choice.\n\n{results}"
    )

## Testing calling the LLM for single sample

In [233]:
def call_llm(system_prompt, conversation_history, model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            *conversation_history  
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

In [232]:
import json
def run_single_text_pipeline(
    text: str,
    previous_classification: str,
    model = "meta-llama/llama-3.1-70b-instruct",
    verbose=True,
):
    """
    Run a multi-step pipeline where each LLM call sees the full conversation history.
    
    Each step appends to the conversation, so later steps have context from earlier ones.
    """
    conversation_history = []
    
    # ============================================================================
    # STEP 1: Generate Summary
    # ============================================================================
    p1 = get_prompt_1(text)
    conversation_history.append({"role": "user", "content": p1})
    print(f"conversation history: {conversation_history}")
    
    summary = call_llm(SYSTEM_PROMPT, conversation_history, model)
    conversation_history.append({"role": "assistant", "content": summary})
    
    if verbose:
        print(f"\n{'='*60}")
        print("STEP 1: SUMMARY")
        print(f"{'='*60}")
        print(summary)
    
    # ============================================================================
    # STEP 2: Climate Litigation Classification
    # ============================================================================
    p2 = get_prompt_2(text, summary)
    conversation_history.append({"role": "user", "content": p2})
    
    cls_raw = call_llm(SYSTEM_PROMPT, conversation_history, model)
    conversation_history.append({"role": "assistant", "content": cls_raw})
    
    if verbose:
        print(f"\n{'='*60}")
        print("STEP 2: CLASSIFICATION")
        print(f"{'='*60}")
        print(cls_raw)
    
    # ============================================================================
    # STEP 3: Verification/Explanation
    # ============================================================================
    p3 = get_prompt_3(text, summary, previous_classification)
    conversation_history.append({"role": "user", "content": p3})
    
    verification = call_llm(SYSTEM_PROMPT, conversation_history, model)
    conversation_history.append({"role": "assistant", "content": verification})
    
    if verbose:
        print(f"\n{'='*60}")
        print("STEP 3: VERIFICATION")
        print(f"{'='*60}")
        print(verification)
    
    # ============================================================================
    # STEP 4: Detailed Breakdown (JSON)
    # ============================================================================
    p4 = get_prompt_4(text, summary)
    conversation_history.append({"role": "user", "content": p4})
    
    raw_breakdown = call_llm(SYSTEM_PROMPT, conversation_history, model)
    conversation_history.append({"role": "assistant", "content": raw_breakdown})
    
    if verbose:
        print(f"\n{'='*60}")
        print("STEP 4: BREAKDOWN (raw)")
        print(f"{'='*60}")
        print(raw_breakdown[:500] + "..." if len(raw_breakdown) > 500 else raw_breakdown)
    
    # Parse and validate JSON
    def _try_parse(js):
        try:
            data = json.loads(js)
            if not isinstance(data, dict):
                return None
            # Check required keys
            required_keys = [
                "parties", "statute_reg_policy", "evidence_quotes",
                "climate_linkage_summary"
            ]
            for k in required_keys:
                if k not in data:
                    return None
            return json.dumps(data, ensure_ascii=False, indent=2)
        except (json.JSONDecodeError, Exception):
            return None
    
    breakdown_json = _try_parse(raw_breakdown)
    
    # Attempt repair if needed
    if breakdown_json is None:
        if verbose:
            print("⚠️  Invalid JSON, attempting repair...")
        
        repair_prompt = (
            "The previous JSON output was invalid. Please return ONLY valid JSON "
            "that matches the required schema. Fix any syntax errors or missing keys. "
            "Do not include explanations, markdown formatting, or additional text.\n\n"
            f"Previous output:\n{raw_breakdown}"
        )
        conversation_history.append({"role": "user", "content": repair_prompt})
        
        repaired = call_llm(SYSTEM_PROMPT, conversation_history, model)
        conversation_history.append({"role": "assistant", "content": repaired})
        
        breakdown_json = _try_parse(repaired)
        
        # Last resort fallback
        if breakdown_json is None:
            if verbose:
                print("⚠️  Repair failed, using empty template")
            breakdown_json = json.dumps({
                "parties": [],
                "case_title_or_docket": None,
                "forum_or_jurisdiction": None,
                "statute_reg_policy": [],
                "procedural_posture": None,
                "dates_or_timeframes": [],
                "monetary_or_quantified": [],
                "climate_linkage_summary": "",
                "climate_linkage_quotes": [],
                "checkable_elements": [],
                "who": None,
                "what": None,
                "where": None,
                "when": None,
                "under_which_rule": None,
                "sought_outcome": None,
                "evidence_quotes": []
            }, ensure_ascii=False, indent=2)
    
    # ============================================================================
    # STEP 5: Specificity Label
    # ============================================================================
    p5 = get_prompt_5(text, summary, previous_classification, breakdown_json)
    conversation_history.append({"role": "user", "content": p5})
    
    label_raw = call_llm(SYSTEM_PROMPT, conversation_history, model)
    conversation_history.append({"role": "assistant", "content": label_raw})
    
    if verbose:
        print(f"\n{'='*60}")
        print("STEP 5: SPECIFICITY")
        print(f"{'='*60}")
        print(label_raw)
    
    # Normalize specificity label
    label = label_raw.strip().lower()
    specificity = label
    
    # ============================================================================
    # Compile full message thread
    # ============================================================================
    full_messages = ""
    for i, msg in enumerate(conversation_history):
        role = msg["role"].upper()
        content = msg["content"]
        full_messages += f"\n{'='*60}\n"
        full_messages += f"{role} MESSAGE {i+1}\n"
        full_messages += f"{'='*60}\n"
        full_messages += f"{content}\n"
    
    if verbose:
        print(f"\n{'='*60}")
        print("PIPELINE COMPLETE")
        print(f"{'='*60}")
        print(f"Classification: {previous_classification}")
        print(f"Specificity: {specificity}")
    
    return {
        "summary": summary,
        "classification": previous_classification,
        "verification": verification,
        "breakdown_json": breakdown_json,
        "specificity_label": specificity,
        "conversation_history": conversation_history,
        "full_messages": full_messages,
    }

In [201]:
sentence_1 = all_results.iloc[25]['text']
classification_1 = all_results.iloc[13]['climate_litigation_binary']
if classification_1 == 1:
    classification_1 = 'yes, this is an instance of climate litigation disclosure'
else:
    classification_1 = 'no, this is not an instance of climate litigation disclosure'


In [175]:
result = run_single_text_pipeline(sentence_1, classification_1)

conversation history: [{'role': 'user', 'content': 'Summarize the following text: If we cannot find adequate sources of water for our use or we are unable to dispose of or recycle the water at a reasonable cost and within applicable environmental rules, our ability to produce natural gas economically and in commercial quantities could be impaired.\nAs part of our drilling and production in shale formations, we use hydraulic fracturing processes. These processes require access to adequate sources of water, which may not be available in proximity to our operations or at certain times of the year. To ensure that we have adequate water available for our operations, we may be required to invest substantial amounts of capital in water pipelines which are used for relatively short periods of time. Alternatively, we may be required to truck water, and we may not be able to contract for sufficient water hauling trucks to meet our needs.\nFurther, we must remove the portion of the water that flo

In [178]:
full_messages = result["full_messages"]

## Testing a pairwise comparison

In [ ]:
def pairwise (idx_1, idx_2, model = "meta-llama/llama-3.1-70b-instruct"):
    text_1 = all_results.iloc[idx_1]['text']
    text_2 = all_results.iloc[idx_2]['text']
    classification_1 = all_results.iloc[idx_1]['climate_litigation_binary']
    classification_2 = all_results.iloc[idx_2]['climate_litigation_binary']
    response_1 = run_single_text_pipeline(text_1, classification_1)
    response_2 = run_single_text_pipeline(text_2, classification_2)
    full_messages_1 = response_1["full_messages"]
    full_messages_2 = response_2["full_messages"]
    prompt_6 = get_prompt_6(full_messages_1, full_messages_2)
    pairwise_result = call_llm(SYSTEM_PROMPT, [{"role": "user", "content": prompt_6}], model)
    print(pairwise_result)
    last_prompt = get_prompt_7(pairwise_result)
    final_result = call_llm(SYSTEM_PROMPT, [{"role": "user", "content": last_prompt}], model)
    print(final_result)

    return final_result


In [203]:
pairwise(1, 2)

conversation history: [{'role': 'user', 'content': "Summarize the following text: In the event we declare a dividend, we will need to repay amounts deferred under the export credit facilities.\nCompliance and Regulatory Risks\nChanges in U.S. or other countries’ foreign travel policy have affected, and may continue to affect our results of operations.\nChanges in U.S. and other countries' foreign policy have in the past and could in the future result in the imposition of travel restrictions or travel bans on persons to certain countries or result in the imposition of travel advisories, warnings, rules, regulations or legislation exposing us to penalties or claims of monetary damages. In addition, some countries have previously adopted restrictions against U.S. travelers. The timing and scope of these changes and regulations can be unpredictable, and they could cause us to cancel scheduled sailings, possibly on short notice, or could result in litigation against us. This, in turn, could

'text 1'

# Testing

We are going to test by seeing which model best maintains transitivity. Because LLMs are famously bad at this, this will give us a good sense as to whether it is doing a good job. 

Transitivity can be understood as: if an LLM says text 1 is more specific than text 2, and that text 2 is more specific than text 3, then it should also say text 1 is more specific than text 3.

To do this, we will give 15 sentences to 10 models. All 15 sentences will be compared against all other sentences. We will ignore needing to test them in opposite order (ie. sentence 1 v sentence 2 and sentence 2 v sentence 1). This means we will have a total of Σ(i) from i=1 to 14 which is 105 combinations. In total, this will give us 1050 combinations. However, we do not need to recompute the single responses each time. So we will do that in isolation and then have a slimmer pairwise comparison function. 

In [248]:
def slimmer_pairwise(breakdown_1, breakdown_2, model = "meta-llama/llama-3.1-70b-instruct"):
    prompt_6 = get_prompt_6(breakdown_1, breakdown_2)
    pairwise_result = call_llm(SYSTEM_PROMPT, [{"role": "user", "content": prompt_6}], model)
    print(pairwise_result)
    last_prompt = get_prompt_7(pairwise_result)
    print(last_prompt)
    final_result = call_llm(SYSTEM_PROMPT, [{"role": "user", "content": last_prompt}], model)
    print(final_result)

    return pairwise_result, final_result

In [207]:
def run_and_save_one_text(idx, folder, model): 
    text = all_results.iloc[idx]['text']
    classification = all_results.iloc[idx]['climate_litigation_binary']
    response = run_single_text_pipeline(text, classification, model)
    full_messages = response["full_messages"]
    # Sanitize model name for filename (replace / and spaces with _)
    safe_model = str(model).replace("/", "_").replace(" ", "_")
    with open(f"{folder}/full_messages_{idx}_{safe_model}.txt", "w") as f:
        f.write(full_messages)
    return full_messages

In [ ]:
model_list = [
    "meta-llama/llama-3.2-1b-instruct", #0.005/M input tokens, $0.01/M output tokens
    "liquid/lfm-7b", #$0.01/M input tokens, $0.01/M output tokens
    "meta-llama/llama-3.2-3b-instruct", #$0.01/M input tokens, $0.02/M output
    "mistralai/mistral-nemo", #$0.01/M input tokens, $0.027/M output tokens
    "liquid/lfm-3b", #$0.02/M input tokens, $0.02/M output tokens
    "meta-llama/llama-3.1-8b-instruct", #$0.019/M input tokens, $0.03/M output tokens
    "google/gemma-3-4b-it", #$0.02/M input tokens, $0.04/M output tokens
    "sao10k/l3-lunaris-8b", #$0.02/M input tokens, $0.05/M output tokens
    "nousresearch/hermes-2-pro-llama-3-8b", #$0.025/M input tokens, $0.04/M output tokens
    "mistralai/mistral-7b-instruct", # $0.028/M input tokens, $0.054/M output tokens
]

In [212]:
import os

output_folder = "../outputs"
os.makedirs(output_folder, exist_ok=True)

for model in model_list:
    print(f"Running model: {model}")
    for idx in range(15):
        print(f"Processing sentence {idx} with model {model}")
        run_and_save_one_text(idx, folder=output_folder, model=model)


Running model: meta-llama/llama-3.2-1b-instruct
Processing sentence 0 with model meta-llama/llama-3.2-1b-instruct
conversation history: [{'role': 'user', 'content': 'Summarize the following text: or other countries’ foreign travel policy have affected, and may continue to affect our results of operations.\nChanges in U.S. and other countries\' foreign policy have in the past and could in the future result in the imposition of travel restrictions or travel bans on persons to certain countries or result in the imposition of travel advisories, warnings, rules, regulations or legislation exposing us to penalties or claims of monetary damages. In addition, some countries have previously adopted restrictions against U.S. travelers. The timing and scope of these changes and regulations can be unpredictable, and they could cause us to cancel scheduled sailings, possibly on short notice, or could result in litigation against us. This, in turn, could decrease our revenue, increase our operating 

Now that we have all the slimmer outputs, we can perform the pairwise comparisons. 

In [215]:
# Search in the output folder for any file name that contains the model name string

model_name = "google/gemma-3-4b-it"
matching_files = [f for f in os.listdir(output_folder) if model_name.replace("/", "_").replace(" ", "_") in f]
print("Matching files:", matching_files)
print(len(matching_files))

Matching files: ['full_messages_14_google_gemma-3-4b-it.txt', 'full_messages_10_google_gemma-3-4b-it.txt', 'full_messages_2_google_gemma-3-4b-it.txt', 'full_messages_6_google_gemma-3-4b-it.txt', 'full_messages_7_google_gemma-3-4b-it.txt', 'full_messages_3_google_gemma-3-4b-it.txt', 'full_messages_11_google_gemma-3-4b-it.txt', 'full_messages_1_google_gemma-3-4b-it.txt', 'full_messages_5_google_gemma-3-4b-it.txt', 'full_messages_9_google_gemma-3-4b-it.txt', 'full_messages_13_google_gemma-3-4b-it.txt', 'full_messages_12_google_gemma-3-4b-it.txt', 'full_messages_4_google_gemma-3-4b-it.txt', 'full_messages_0_google_gemma-3-4b-it.txt', 'full_messages_8_google_gemma-3-4b-it.txt']
15


In [ ]:
#open these matching files and save them to a dataframe where the first column is the file name and the second column is the full message

rows = []
for file in matching_files:
    with open(os.path.join(output_folder, file), "r") as f:
        rows.append({"file_name": file, "full_message": f.read()})
full_messages_df = pd.DataFrame(rows, columns=["file_name", "full_message"])

,file_name,full_message
0,full_messages_14_google_gemma-3-4b-it.txt,\n============================================...
1,full_messages_10_google_gemma-3-4b-it.txt,\n============================================...
2,full_messages_2_google_gemma-3-4b-it.txt,\n============================================...
3,full_messages_6_google_gemma-3-4b-it.txt,\n============================================...
4,full_messages_7_google_gemma-3-4b-it.txt,\n============================================...


In [381]:
#wrap this in a function 
def load_full_messages(model_name, output_folder):
    matching_files = [f for f in os.listdir(output_folder) if model_name.replace("/", "_").replace(" ", "_") in f]
    rows = []
    for file in matching_files:
        with open(os.path.join(output_folder, file), "r") as f:
            rows.append({"file_name": file, "full_message": f.read()})
    return pd.DataFrame(rows, columns=["file_name", "full_message"])

In [220]:
gemma_df = load_full_messages("google/gemma-3-4b-it")

In [243]:
# Now, within the given df, perform pairwise comparisons and save results in a dataframe

def by_model_pairwise(df):
    results = []
    pairwise_results_df = pd.DataFrame(columns=["idx_1", "idx_2", "reasoning", "result"])

    for idx_1 in range(15):
        for idx_2 in range(idx_1 + 1, 15):
            # Check if this pair has already been computed in this run
            already_done = (
                ((pairwise_results_df["idx_1"] == idx_1) & (pairwise_results_df["idx_2"] == idx_2)).any()
            )
            if already_done:
                continue
            print(f"Comparing sentence {idx_1} with sentence {idx_2}")
            # Extract the full message text for each index
            breakdown_1 = df[df['file_name'].str.contains(f'full_messages_{idx_1}_')]['full_message'].values[0]
            breakdown_2 = df[df['file_name'].str.contains(f'full_messages_{idx_2}_')]['full_message'].values[0]
            full_result, binary_result = slimmer_pairwise(breakdown_1, breakdown_2, model="google/gemma-3-4b-it")
            results.append({
                "idx_1": idx_1,
                "idx_2": idx_2,
                "reasoning" : full_result,
                "result": binary_result
            })
            # Update the dataframe after each comparison
            pairwise_results_df = pd.DataFrame(results, columns=["idx_1", "idx_2", "reasoning", "result"])
    
    return pairwise_results_df

In [245]:
pairwise_results_dfs = {}
for model in model_list:
    model_df = load_full_messages(model)
    print(f"Loaded {len(model_df)} rows for model: {model}")
    if len(model_df) != 15:
        print(f"Error: load_full_messages for model '{model}' returned {len(model_df)} rows instead of 15.")
        continue
    pairwise_results_df = by_model_pairwise(model_df)
    pairwise_results_dfs[model] = pairwise_results_df

Loaded 15 rows for model: meta-llama/llama-3.2-1b-instruct
Comparing sentence 0 with sentence 1
The provided text primarily discusses potential regulatory risks and business impacts related to foreign travel policies and climate change, rather than a specific legal action or claim. While it mentions the *possibility* of climate change-related litigation, it lacks the specific details (parties, dates, legal arguments, etc.) necessary to classify it as a concrete climate litigation case.

FINAL: text 1
text 1

Comparing sentence 0 with sentence 2
The breakdown JSON provides a detailed level of specificity, identifying key statutes, regulations, and potential parties involved in the discussion of climate-related risks. It includes specific dates (2024) and clearly outlines the potential for litigation. The climate linkage is explicitly stated, and the elements are directly traceable to climate change concerns.

FINAL: text 2
text 2

Comparing sentence 0 with sentence 3
The breakdown provi

In [246]:
pairwise_results_dfs

{'meta-llama/llama-3.2-1b-instruct':      idx_1  idx_2                                          reasoning    result
 0        0      1  The provided text primarily discusses potentia...  text 1\n
 1        0      2  The breakdown JSON provides a detailed level o...  text 2\n
 2        0      3  The breakdown provides a relatively detailed l...  text 2\n
 3        0      4  The text explicitly details lawsuits filed by ...  text 2\n
 4        0      5  The text primarily focuses on potential risks ...  text 2\n
 ..     ...    ...                                                ...       ...
 100     11     13  The provided text explicitly details ongoing c...  text 1\n
 101     11     14  The text primarily discusses regulatory change...  text 2\n
 102     12     13  detailed disclosure\n\nThe breakdown includes ...  text 2\n
 103     12     14  The text discusses regulatory actions related ...  text 2\n
 104     13     14  The breakdown identifies the core issue as the...  text 2\n
 
 [

In [267]:
def check_invalid_results(df_list):
    invalid_entries = []
    for model, df in df_list.items():
        for idx, row in df.iterrows():
            result = str(row['result']).strip().lower()
            if result not in ['text 1', 'text 2', 'neither']:
                invalid_entries.append({
                    'model': model,
                    'idx_1': row['idx_1'],
                    'idx_2': row['idx_2'],
                    'result': row['result']
                })
    return invalid_entries

invalid_results = check_invalid_results(pairwise_results_dfs)

In [253]:
from collections import Counter

# Count invalid results per model
invalid_counts = Counter()
for entry in invalid_results:
    model = entry['model']
    invalid_counts[model] += 1

# Print the percentage of invalid results for each model (out of 105)
for model, count in invalid_counts.items():
    percent = (count / 105) * 100
    print(f"Model: {model}, Invalid results: {count} ({percent:.2f}%)")


Model: meta-llama/llama-3.2-1b-instruct, Invalid results: 22 (20.95%)
Model: liquid/lfm-7b, Invalid results: 2 (1.90%)
Model: meta-llama/llama-3.2-3b-instruct, Invalid results: 1 (0.95%)
Model: mistralai/mistral-nemo, Invalid results: 24 (22.86%)
Model: liquid/lfm-3b, Invalid results: 40 (38.10%)
Model: meta-llama/llama-3.1-8b-instruct, Invalid results: 23 (21.90%)
Model: google/gemma-3-4b-it, Invalid results: 8 (7.62%)
Model: sao10k/l3-lunaris-8b, Invalid results: 5 (4.76%)
Model: nousresearch/hermes-2-pro-llama-3-8b, Invalid results: 1 (0.95%)
Model: mistralai/mistral-7b-instruct, Invalid results: 24 (22.86%)


In [ ]:
# Overwrite invalid results in pairwise_results_dfs with rerun slimmer_pairwise results

for entry in invalid_results:
    model = entry['model']
    print(f"Model: {model}")
    idx_1 = entry['idx_1']
    idx_2 = entry['idx_2']
    # Get the model_df used for this model
    model_df = load_full_messages(model)
    # Get the texts for idx_1 and idx_2
    text_1 = model_df.loc[model_df.index == idx_1, 'full_message'].values[0]
    text_2 = model_df.loc[model_df.index == idx_2, 'full_message'].values[0]
    # Run slimmer_pairwise on the two texts
    result, final_result = slimmer_pairwise(text_1, text_2)
    # Overwrite the result in the pairwise_results_dfs
    pairwise_df = pairwise_results_dfs[model]
    mask = (pairwise_df['idx_1'] == idx_1) & (pairwise_df['idx_2'] == idx_2)
    # Use .at to set the value for the single row
    idxs = pairwise_df.index[mask]
    if len(idxs) == 1:
        pairwise_results_dfs[model].at[idxs[0], 'result'] = final_result
        pairwise_results_dfs[model].at[idxs[0], 'reasoning'] = result
    elif len(idxs) == 0:
        print(f"Warning: No matching row found for model={model}, idx_1={idx_1}, idx_2={idx_2}")
    else:
        print(f"Warning: Multiple matching rows found for model={model}, idx_1={idx_1}, idx_2={idx_2}")


Model: meta-llama/llama-3.2-1b-instruct
Text 1 provides more specific details about climate litigation, including the mention of civil litigation and regulatory focus on global climate change, as well as specific regulations such as the Clean Air Act and the Paris Climate Accord. In contrast, Text 2 only mentions the potential for increased exposure to climate change-related litigation, but does not provide specific details about the litigation itself.

FINAL: text 1
In the following content, extract the final decision. Return ONLY 'text 1' or 'text 2' or 'neither' (no other text). If absent, infer from the last explicit choice.

Text 1 provides more specific details about climate litigation, including the mention of civil litigation and regulatory focus on global climate change, as well as specific regulations such as the Clean Air Act and the Paris Climate Accord. In contrast, Text 2 only mentions the potential for increased exposure to climate change-related litigation, but does not

In [260]:
from collections import Counter

# Count invalid results per model
invalid_counts = Counter()
for entry in invalid_results:
    model = entry['model']
    invalid_counts[model] += 1

# Print the percentage of invalid results for each model (out of 105)
for model, count in invalid_counts.items():
    percent = (count / 105) * 100
    print(f"Model: {model}, Invalid results: {count} ({percent:.2f}%)")


Model: meta-llama/llama-3.2-1b-instruct, Invalid results: 22 (20.95%)
Model: liquid/lfm-7b, Invalid results: 2 (1.90%)
Model: meta-llama/llama-3.2-3b-instruct, Invalid results: 1 (0.95%)
Model: mistralai/mistral-nemo, Invalid results: 24 (22.86%)
Model: liquid/lfm-3b, Invalid results: 40 (38.10%)
Model: meta-llama/llama-3.1-8b-instruct, Invalid results: 23 (21.90%)
Model: google/gemma-3-4b-it, Invalid results: 8 (7.62%)
Model: sao10k/l3-lunaris-8b, Invalid results: 5 (4.76%)
Model: nousresearch/hermes-2-pro-llama-3-8b, Invalid results: 1 (0.95%)
Model: mistralai/mistral-7b-instruct, Invalid results: 24 (22.86%)


In [263]:
# Retroactively fix any tuples that were stored in the 'result' column
fixed_count = 0
for model, df in pairwise_results_dfs.items():
    for idx, row in df.iterrows():
        result_val = row['result']
        # Check if it's a tuple
        if isinstance(result_val, tuple):
            fixed_count += 1
            # Extract the second element (final_result) for 'result' column
            # Extract the first element (pairwise_result) for 'reasoning' column
            pairwise_result, final_result = result_val
            pairwise_results_dfs[model].at[idx, 'result'] = final_result
            pairwise_results_dfs[model].at[idx, 'reasoning'] = pairwise_result
            print(f"Fixed tuple in model {model}, row {idx}: {final_result}")

print(f"\nTotal tuples fixed: {fixed_count}")

Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 7: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 9: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 11: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 20: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 24: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 32: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 36: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 43: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 45: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 47: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 53: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 57: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 64: text 1
Fixed tuple in model meta-llama/llama-3.2-1b-instruct, row 66: neither
Fixed tuple in model 

In [268]:
# Recompute invalid results after the fix
invalid_results_after_fix = check_invalid_results(pairwise_results_dfs)

# Count invalid results per model
invalid_counts_after_fix = Counter()
for entry in invalid_results_after_fix:
    model = entry['model']
    invalid_counts_after_fix[model] += 1

# Print the percentage of invalid results for each model (out of 105)
print("AFTER RETROACTIVE FIX:")
print("="*60)
for model in model_list:
    count = invalid_counts_after_fix.get(model, 0)
    percent = (count / 105) * 100
    print(f"Model: {model}")
    print(f"  Invalid results: {count} ({percent:.2f}%)")
    print()

print(f"Total invalid results across all models: {len(invalid_results_after_fix)}")


AFTER RETROACTIVE FIX:
Model: meta-llama/llama-3.2-1b-instruct
  Invalid results: 0 (0.00%)

Model: liquid/lfm-7b
  Invalid results: 0 (0.00%)

Model: meta-llama/llama-3.2-3b-instruct
  Invalid results: 0 (0.00%)

Model: mistralai/mistral-nemo
  Invalid results: 0 (0.00%)

Model: liquid/lfm-3b
  Invalid results: 0 (0.00%)

Model: meta-llama/llama-3.1-8b-instruct
  Invalid results: 0 (0.00%)

Model: google/gemma-3-4b-it
  Invalid results: 0 (0.00%)

Model: sao10k/l3-lunaris-8b
  Invalid results: 0 (0.00%)

Model: nousresearch/hermes-2-pro-llama-3-8b
  Invalid results: 0 (0.00%)

Model: mistralai/mistral-7b-instruct
  Invalid results: 0 (0.00%)

Total invalid results across all models: 0


_____

## Examining Test Results

Now we have our pairwise comparisons for 15 sentences across 10 models, all healed where they failed initially. Now we need to analyse the variety and transitivity 

In [269]:
pairwise_results_dfs['mistralai/mistral-7b-instruct']

,idx_1,idx_2,reasoning,result
0,0,1,Text 1 provides more specific details about cl...,text 1
1,0,2,The text discusses potential future climate re...,text 2\n
2,0,3,Text 1 provides more specific details about cl...,text 1
3,0,4,Text 1 provides more specific details about th...,text 1
4,0,5,Text 2 is more specific. While Text 1 discusse...,text 2
...,...,...,...,...
100,11,13,detailed disclosure\n\nThe breakdown provides ...,text 2\n
101,11,14,The breakdown provides specific details regard...,text 2\n
102,12,13,detailed disclosure\n\nThe breakdown provides ...,text 2\n
103,12,14,detailed disclosure\n\nThe breakdown provides ...,text 2\n


In [275]:
# The result should indicate the index that won, rather than the text number.
# For example, if the result is "text 1", replace it with the value in idx_1; if "text 2", use idx_2.

def convert_result_to_index(df):
    def get_winner(row):
        if isinstance(row['result'], str):
            if 'text 1' in row['result'].lower():
                return row['idx_1']
            elif 'text 2' in row['result'].lower():
                return row['idx_2']
        return None  # or np.nan if you prefer
    df = df.copy()
    df['winner_idx'] = df.apply(get_winner, axis=1)
    return df

In [276]:
for model, df in pairwise_results_dfs.items():
    pairwise_results_dfs[model] = convert_result_to_index(df)


In [277]:
pairwise_results_dfs['mistralai/mistral-7b-instruct']

,idx_1,idx_2,reasoning,result,winner_idx
0,0,1,Text 1 provides more specific details about cl...,text 1,0.0
1,0,2,The text discusses potential future climate re...,text 2\n,2.0
2,0,3,Text 1 provides more specific details about cl...,text 1,0.0
3,0,4,Text 1 provides more specific details about th...,text 1,0.0
4,0,5,Text 2 is more specific. While Text 1 discusse...,text 2,5.0
...,...,...,...,...,...
100,11,13,detailed disclosure\n\nThe breakdown provides ...,text 2\n,13.0
101,11,14,The breakdown provides specific details regard...,text 2\n,14.0
102,12,13,detailed disclosure\n\nThe breakdown provides ...,text 2\n,13.0
103,12,14,detailed disclosure\n\nThe breakdown provides ...,text 2\n,14.0


In [305]:
from itertools import combinations

def check_transitivity(df):
    """
    Check transitivity violations in pairwise comparison data.
    
    Args:
        df: DataFrame with columns 'idx_1', 'idx_2', and 'winner_idx'
    
    Returns:
        dict with violation statistics and details
    """
    
    # Create a dictionary to store comparison results
    # Key: (idx_a, idx_b), Value: winner_idx
    comparisons = {}
    
    for _, row in df.iterrows():
        idx_a, idx_b = row['idx_1'], row['idx_2']
        winner = row['winner_idx']
        
        comparisons[(idx_a, idx_b)] = winner
    
    # Get all unique indices
    all_indices = set(df['idx_1'].unique()) | set(df['idx_2'].unique())
    
    # Check transitivity for all triplets
    violations = []
    total_triplets = 0
    
    for a, b, c in combinations(sorted(all_indices), 3):
        # Check if all three pairwise comparisons exist
        if (a, b) not in comparisons or (a, c) not in comparisons or (b, c) not in comparisons:
            print('This happened. This did not happen in testing originally, so pay attention to it.')
            continue
        
        total_triplets += 1
        
        winner_ab = comparisons[(a, b)]
        winner_ac = comparisons[(a, c)]
        winner_bc = comparisons[(b, c)]
        
        # Skip if any comparison resulted in 'neither' (represented as NaN or string). We should only have 'neither' if one of the texts is not climate litigation.
        if pd.isna(winner_ab) or pd.isna(winner_ac) or pd.isna(winner_bc):
            continue
        
        # Check all transitivity patterns
        # If a > b and a > c, we can't infer b vs c (both valid)
        # If a > b and b > c, then a must > c
        if winner_ab == a and winner_bc == b:
            if winner_ac != a:
                violations.append({
                    'triplet': (a, b, c),
                    'pattern': f"{a}>{b}, {b}>{c}",
                    'expected': f"{a}>{c}",
                    'actual': f"{int(winner_ac)}>{c if winner_ac == c else a}"
                })
        
        # If a > b and c > a, then c must > b
        elif winner_ab == a and winner_ac == c:
            if winner_bc != c:
                violations.append({
                    'triplet': (a, b, c),
                    'pattern': f"{a}>{b}, {c}>{a}",
                    'expected': f"{c}>{b}",
                    'actual': f"{int(winner_bc)}>{b if winner_bc == b else c}"
                })
        
        # If b > a and a > c, then b must > c
        elif winner_ab == b and winner_ac == a:
            if winner_bc != b:
                violations.append({
                    'triplet': (a, b, c),
                    'pattern': f"{b}>{a}, {a}>{c}",
                    'expected': f"{b}>{c}",
                    'actual': f"{int(winner_bc)}>{c if winner_bc == c else b}"
                })
        
        # If b > a and c > b, then c must > a
        elif winner_ab == b and winner_bc == c:
            if winner_ac != c:
                violations.append({
                    'triplet': (a, b, c),
                    'pattern': f"{b}>{a}, {c}>{b}",
                    'expected': f"{c}>{a}",
                    'actual': f"{int(winner_ac)}>{a if winner_ac == a else c}"
                })
        
        # If c > a and a > b, then c must > b
        elif winner_ac == c and winner_ab == a:
            if winner_bc != c:
                violations.append({
                    'triplet': (a, b, c),
                    'pattern': f"{c}>{a}, {a}>{b}",
                    'expected': f"{c}>{b}",
                    'actual': f"{int(winner_bc)}>{b if winner_bc == b else c}"
                })
        
        # If c > a and b > c, then b must > a
        elif winner_ac == c and winner_bc == b:
            if winner_ab != b:
                violations.append({
                    'triplet': (a, b, c),
                    'pattern': f"{c}>{a}, {b}>{c}",
                    'expected': f"{b}>{a}",
                    'actual': f"{int(winner_ab)}>{a if winner_ab == a else b}"
                })
    
    return {
        'total_triplets_checked': total_triplets,
        'violations': violations,
        'num_violations': len(violations),
        'transitivity_rate': 1 - (len(violations) / total_triplets) if total_triplets > 0 else 1.0
    }

In [279]:
for model, df in pairwise_results_dfs.items():
    results = check_transitivity(df)
    print(f"Model: {model}")
    print(f"Transitivity Rate: {results['transitivity_rate']:.2%}")
    print(f"Violations: {results['num_violations']} out of {results['total_triplets_checked']} triplets")
    


Model: meta-llama/llama-3.2-1b-instruct
Transitivity Rate: 95.60%
Violations: 20 out of 455 triplets
Model: liquid/lfm-7b
Transitivity Rate: 94.07%
Violations: 27 out of 455 triplets
Model: meta-llama/llama-3.2-3b-instruct
Transitivity Rate: 91.87%
Violations: 37 out of 455 triplets
Model: mistralai/mistral-nemo
Transitivity Rate: 86.15%
Violations: 63 out of 455 triplets
Model: liquid/lfm-3b
Transitivity Rate: 87.25%
Violations: 58 out of 455 triplets
Model: meta-llama/llama-3.1-8b-instruct
Transitivity Rate: 91.65%
Violations: 38 out of 455 triplets
Model: google/gemma-3-4b-it
Transitivity Rate: 98.68%
Violations: 6 out of 455 triplets
Model: sao10k/l3-lunaris-8b
Transitivity Rate: 96.70%
Violations: 15 out of 455 triplets
Model: nousresearch/hermes-2-pro-llama-3-8b
Transitivity Rate: 98.68%
Violations: 6 out of 455 triplets
Model: mistralai/mistral-7b-instruct
Transitivity Rate: 93.19%
Violations: 31 out of 455 triplets


In [290]:
#now I need to run these pairwise comparisons using a SOTA model 
SOTA_MODEL = 'openai/gpt-5-chat'

output_folder = "../SOTA_outputs"
os.makedirs(output_folder, exist_ok=True)

print(f"Running model: {SOTA_MODEL}")
for idx in range(15):
    print(f"Processing sentence {idx} with model {SOTA_MODEL}")
    run_and_save_one_text(idx, folder=output_folder, model=SOTA_MODEL)


Running model: openai/gpt-5-chat
Processing sentence 0 with model openai/gpt-5-chat
conversation history: [{'role': 'user', 'content': 'Summarize the following text: or other countries’ foreign travel policy have affected, and may continue to affect our results of operations.\nChanges in U.S. and other countries\' foreign policy have in the past and could in the future result in the imposition of travel restrictions or travel bans on persons to certain countries or result in the imposition of travel advisories, warnings, rules, regulations or legislation exposing us to penalties or claims of monetary damages. In addition, some countries have previously adopted restrictions against U.S. travelers. The timing and scope of these changes and regulations can be unpredictable, and they could cause us to cancel scheduled sailings, possibly on short notice, or could result in litigation against us. This, in turn, could decrease our revenue, increase our operating costs and otherwise impair our

In [292]:
#now we need to do the pairwise comparisons for the SOTA model 
model_df = load_full_messages(SOTA_MODEL)
print(f"Loaded {len(model_df)} rows for model: {SOTA_MODEL}")
if len(model_df) != 15:
    print(f"Error: load_full_messages for model '{SOTA_MODEL}' returned {len(model_df)} rows instead of 15.")
SOTA_results = by_model_pairwise(model_df)

Loaded 15 rows for model: openai/gpt-5-chat
Comparing sentence 0 with sentence 1
The text primarily discusses potential operational and financial risks associated with changing travel and environmental policies, with a secondary mention of potential climate litigation exposure. However, the disclosure is highly anticipatory and lacks specific details regarding any actual or threatened legal actions.

FINAL: text 1
In the following content, extract the final decision. Return ONLY 'text 1' or 'text 2' or 'neither' (no other text). If absent, infer from the last explicit choice.

The text primarily discusses potential operational and financial risks associated with changing travel and environmental policies, with a secondary mention of potential climate litigation exposure. However, the disclosure is highly anticipatory and lacks specific details regarding any actual or threatened legal actions.

FINAL: text 1
text 1

Comparing sentence 0 with sentence 2
The text primarily focuses on the 

In [296]:
# Let's see if there are any invalid responses for the SOTA results.
def check_invalid_results_single(df):
    """
    Checks for invalid results in a single pairwise results DataFrame.
    Returns a dictionary with counts and indices of invalid results.
    """
    invalid_indices = []
    for idx, row in df.iterrows():
        result = str(row.get('result', '')).strip().lower()
        if result not in ['text 1', 'text 2']:
            invalid_indices.append(idx)
    return {
        'num_invalid': len(invalid_indices),
        'invalid_indices': invalid_indices
    }

invalid_results = check_invalid_results_single(SOTA_results)
print(invalid_results)

#let's see these invalid results 
# Show the rows in SOTA_results where the result is invalid
invalid_indices = invalid_results['invalid_indices']
if invalid_indices:
    print("Invalid result rows:")
    display(SOTA_results.loc[invalid_indices])
else:
    print("No invalid result rows found.")



{'num_invalid': 26, 'invalid_indices': [2, 3, 4, 5, 6, 10, 12, 13, 15, 18, 23, 26, 27, 28, 29, 30, 31, 35, 37, 38, 60, 63, 69, 86, 95, 98]}
Invalid result rows:


,idx_1,idx_2,reasoning,result
2,0,3,Rationale: The text details financial accounti...,"Okay, I understand. Please provide the text.\n"
3,0,4,Rationale: The text primarily discusses genera...,"Okay, I understand. Please provide the text.\n"
4,0,5,Rationale: The text primarily discusses genera...,"Okay, please provide the text. I’m ready to an..."
5,0,6,Rationale: The text identifies CNX as a party ...,"Okay, I understand. Please provide the text.\n"
6,0,7,Rationale: The text mentions potential litigat...,"Okay, I understand. Please provide the text.\n"
10,0,11,Rationale: The text primarily discusses genera...,"Okay, please provide the text. I’m ready to an..."
12,0,13,Rationale: The disclosure identifies potential...,"Okay, please provide the text. I’m ready to an..."
13,0,14,Rationale: The text identifies potential futur...,"Okay, I understand. Please provide the text.\n"
15,1,3,Rationale: The text describes financial report...,"Okay, I understand. Please provide the text.\n"
18,1,6,Rationale: The text identifies the existence o...,"Okay, I understand. Please provide the text.\n"


In [300]:
#for the invalid results, search within the reasoning column and see if it says FINAL: text 1 or FINAL: text 2. If it does, replace the result with the correct text.
for idx, row in SOTA_results.iterrows():
    if idx in invalid_indices:
        reasoning = row['reasoning']
        if 'FINAL: text 1' in reasoning:
            SOTA_results.at[idx, 'result'] = 'text 1'
        elif 'FINAL: text 2' in reasoning:
            SOTA_results.at[idx, 'result'] = 'text 2'


In [301]:
invalid_results = check_invalid_results_single(SOTA_results)

In [302]:
invalid_results

{'num_invalid': 0, 'invalid_indices': []}

In [307]:
#ok now let's check the transitivity 
SOTA_results = convert_result_to_index(SOTA_results)
results = check_transitivity(SOTA_results)
print(f"Model: {SOTA_MODEL}")
print(f"Transitivity Rate: {results['transitivity_rate']:.2%}")
print(f"Violations: {results['num_violations']} out of {results['total_triplets_checked']} triplets")
    


Model: openai/gpt-5-chat
Transitivity Rate: 89.89%
Violations: 46 out of 455 triplets


Since this actually underperformed the cheaper models, we are not going to use this as a test comparison

Isntead, we are going to take the top three performers from the cheaper models and test their variability

____

In [310]:
three_models = ['google/gemma-3-4b-it', 'nousresearch/hermes-2-pro-llama-3-8b', 'sao10k/l3-lunaris-8b']

# For each model, treat it as the "gold standard" and compare the other two to it
for i, gold_model in enumerate(three_models):
    gold_df = pairwise_results_dfs[gold_model].reset_index(drop=True)
    print(f"\nUsing {gold_model} as the gold standard:")
    for j, compare_model in enumerate(three_models):
        if gold_model == compare_model:
            continue
        compare_df = pairwise_results_dfs[compare_model].reset_index(drop=True)
        # Compare the 'winner_idx' columns
        if len(gold_df) != len(compare_df):
            print(f"  WARNING: {compare_model} has {len(compare_df)} rows, but {gold_model} has {len(gold_df)} rows.")
            min_len = min(len(gold_df), len(compare_df))
            gold_winners = gold_df['winner_idx'][:min_len]
            compare_winners = compare_df['winner_idx'][:min_len]
        else:
            gold_winners = gold_df['winner_idx']
            compare_winners = compare_df['winner_idx']
        agreement = (gold_winners == compare_winners).mean()
        print(f"  Agreement with {compare_model}: {agreement:.2%} ({(gold_winners == compare_winners).sum()}/{len(gold_winners)})")



Using google/gemma-3-4b-it as the gold standard:
  Agreement with nousresearch/hermes-2-pro-llama-3-8b: 96.19% (101/105)
  Agreement with sao10k/l3-lunaris-8b: 90.48% (95/105)

Using nousresearch/hermes-2-pro-llama-3-8b as the gold standard:
  Agreement with google/gemma-3-4b-it: 96.19% (101/105)
  Agreement with sao10k/l3-lunaris-8b: 88.57% (93/105)

Using sao10k/l3-lunaris-8b as the gold standard:
  Agreement with google/gemma-3-4b-it: 90.48% (95/105)
  Agreement with nousresearch/hermes-2-pro-llama-3-8b: 88.57% (93/105)


So, from that, I see that gemma and nousresearch largely agree.

In [348]:
output_folder = "../outputs"

**This is really bad!! I need to fix this so output folder isn't hard coded in any of my functions because I have two output functions**

In [397]:
#for gemma and nousresearch, let's do a test of inverse comparisons
def by_model_pairwise_reversed(existing_results_df, model):
    import json
    reversed_results = []
    df = load_full_messages(model, output_folder = '../outputs/single/')
    # Get all unique pairs from existing results
    for _, row in existing_results_df.iterrows():
        idx_1 = row['idx_1']
        idx_2 = row['idx_2']
        
        print(f"Comparing sentence {idx_2} with sentence {idx_1} (reversed)")
        
        try:
            # Extract the full message text - NOTE THE REVERSED ORDER
            breakdown_1 = df[df['file_name'].str.contains(f'full_messages_{idx_2}_')]['full_message'].values[0]
            breakdown_2 = df[df['file_name'].str.contains(f'full_messages_{idx_1}_')]['full_message'].values[0]
            
            full_result, binary_result = slimmer_pairwise(breakdown_1, breakdown_2, model=model)
            reversed_results.append({
                "idx_1": idx_2,  # Reversed
                "idx_2": idx_1,  # Reversed
                "reasoning": full_result,
                "result": binary_result,
                "model": model
            })
        except json.JSONDecodeError as e:
            print(f"  Skipping pair ({idx_2}, {idx_1}) due to JSON error: {e}")
            continue
        except Exception as e:
            # Optionally, you can skip any other error as well, or re-raise
            print(f"  Skipping pair ({idx_2}, {idx_1}) due to error: {e}")
            continue
    
    return pd.DataFrame(reversed_results)

In [352]:
gemma_reversed = by_model_pairwise_reversed(pairwise_results_dfs['google/gemma-3-4b-it'], 'google/gemma-3-4b-it')

Comparing sentence 1 with sentence 0 (reversed)
Text 2 is more specific due to its detailed discussion of specific regulations (EU Fit for 55, CII, IMO Sulfur Limit) and their potential financial impacts. Text 1 primarily focuses on broader risks and lacks the granular detail necessary to be considered highly specific.

FINAL: text 2
In the following content, extract the final decision. Return ONLY 'text 1' or 'text 2' or 'neither' (no other text). If absent, infer from the last explicit choice.

Text 2 is more specific due to its detailed discussion of specific regulations (EU Fit for 55, CII, IMO Sulfur Limit) and their potential financial impacts. Text 1 primarily focuses on broader risks and lacks the granular detail necessary to be considered highly specific.

FINAL: text 2
text 2

Comparing sentence 2 with sentence 0 (reversed)
The text primarily focuses on geopolitical risks related to foreign travel policy and the increasing regulatory pressure from climate change, specifically

In [353]:
gemma_reversed = convert_result_to_index(gemma_reversed)
gemma_reversed


,idx_1,idx_2,reasoning,result,model,winner_idx
0,1,0,Text 2 is more specific due to its detailed di...,text 2\n,google/gemma-3-4b-it,0.0
1,2,0,The text primarily focuses on geopolitical ris...,text 2\n,google/gemma-3-4b-it,0.0
2,3,0,The text primarily discusses geopolitical risk...,text 2\n,google/gemma-3-4b-it,0.0
3,4,0,The text primarily focuses on geopolitical ris...,text 2\n,google/gemma-3-4b-it,0.0
4,5,0,The text primarily focuses on geopolitical ris...,text 2\n,google/gemma-3-4b-it,0.0
...,...,...,...,...,...,...
100,13,11,"```json\n{\n ""parties"": [\n ""various state...",text 2\n,google/gemma-3-4b-it,11.0
101,14,11,Text 2 is significantly more specific than Tex...,text 2\n,google/gemma-3-4b-it,11.0
102,13,12,Text 2 demonstrates a significantly higher lev...,text 2\n,google/gemma-3-4b-it,12.0
103,14,12,The text exhibits a high degree of specificity...,text 2\n,google/gemma-3-4b-it,12.0


In [319]:
gemma_forward = pairwise_results_dfs['google/gemma-3-4b-it']

In [354]:
# Compare gemma_forward and gemma_reversed: see how often winner_idx is the same, give a percentage

# Merge on (idx_1, idx_2) <-> (idx_2, idx_1) to align forward and reversed
merged = pd.merge(
    gemma_forward,
    gemma_reversed,
    left_on=['idx_1', 'idx_2'],
    right_on=['idx_2', 'idx_1'],
    suffixes=('_forward', '_reversed')
)

# Compare winner_idx
same_winner = merged['winner_idx_forward'] == merged['winner_idx_reversed']
percent_same = same_winner.mean() * 100

print(f"Percentage of pairs where winner_idx is the same in forward and reversed: {percent_same:.2f}%")


Percentage of pairs where winner_idx is the same in forward and reversed: 3.81%


This is terrible!! It is basically just choosing #2 every time, and we are not seeing any agreement between the forward and reverse. 

I think I need to do this for all the models

In [355]:
# Count the percentage of times each original model predicts "text 1", "text 2", or "neither"
model_percentages = {}
for model, df in pairwise_results_dfs.items():
    # Normalize result column to strip whitespace and lowercase
    results = df['result'].astype(str).str.strip().str.lower()
    total = len(results)
    text1_count = ((results == 'text 1') | (results == 'text 1\n')).sum()
    text2_count = ((results == 'text 2') | (results == 'text 2\n')).sum()
    neither_count = total - text1_count - text2_count
    model_percentages[model] = {
        'text 1': round(100 * text1_count / total, 2) if total > 0 else 0,
        'text 2': round(100 * text2_count / total, 2) if total > 0 else 0,
        'neither': round(100 * neither_count / total, 2) if total > 0 else 0
    }

model_counts_df = pd.DataFrame.from_dict(model_percentages, orient='index').reset_index().rename(columns={'index': 'Model'})


In [337]:
model_counts_df

,Model,text 1,text 2,neither
0,meta-llama/llama-3.2-1b-instruct,24.76,72.38,2.86
1,liquid/lfm-7b,9.52,90.48,0.00
2,meta-llama/llama-3.2-3b-instruct,15.24,84.76,0.00
3,mistralai/mistral-nemo,17.14,79.05,3.81
4,liquid/lfm-3b,35.24,64.76,0.00
5,meta-llama/llama-3.1-8b-instruct,11.43,79.05,9.52
6,google/gemma-3-4b-it,2.86,97.14,0.00
7,sao10k/l3-lunaris-8b,7.62,91.43,0.95
8,nousresearch/hermes-2-pro-llama-3-8b,2.86,97.14,0.00
9,mistralai/mistral-7b-instruct,23.81,75.24,0.95


_____

## Working with reversed results

In [395]:
import glob

# Dictionary to hold reversed dataframes
pairwise_results_dfs_reversed = {}

# Get all reversed CSV files in the output directory
reversed_files = glob.glob("../outputs/reversed/pairwise_results_dfs_*_reversed.csv")

for fpath in reversed_files:
    # Extract model name from the filename
    fname = fpath.split('/')[-1]
    model_name = fname.replace('pairwise_results_dfs_', '').replace('_reversed.csv', '').replace('_', '/')
    # Some models may use underscores instead of slashes or dashes, revert what was done in save path
    # If your model naming scheme is more complex, improve this part
    model_name = model_name.replace('//', '/').replace('--', '-')
    try:
        df = pd.read_csv(fpath)
        pairwise_results_dfs_reversed[model_name] = df
    except Exception as e:
        print(f"Error loading {fname}: {e}")


Error loading pairwise_results_dfs_sao10k_l3_lunaris_8b_reversed.csv: No columns to parse from file
Error loading pairwise_results_dfs_mistralai_mistral_7b_instruct_reversed.csv: No columns to parse from file
Error loading pairwise_results_dfs_google_gemma_3_4b_it_reversed.csv: No columns to parse from file
Error loading pairwise_results_dfs_meta_llama_llama_3.1_8b_instruct_reversed.csv: No columns to parse from file


In [403]:
# Run for all models in pairwise_results_dfs, but skip models that already have 105 rows in their reversed dataframe

for model, df in pairwise_results_dfs.items():
    # If already processed and has 105 rows, skip
    reversed_df = pairwise_results_dfs_reversed.get(model)
    if reversed_df is not None and len(reversed_df) == 105:
        print(f"Skipping {model} as reversed results already have 105 rows.")
        continue
    if 'nous' in model: 
        print("skipping nous model")
        continue
    print(f"Processing {model}")
    try:
        pairwise_results_dfs_reversed[model] = by_model_pairwise_reversed(df, model)
    except Exception as e:
        print(f"Skipping model {model} due to error: {e}")
        continue

pairwise_results_dfs_reversed


Skipping meta-llama/llama-3.2-1b-instruct as reversed results already have 105 rows.
Skipping liquid/lfm-7b as reversed results already have 105 rows.
Skipping meta-llama/llama-3.2-3b-instruct as reversed results already have 105 rows.
Skipping mistralai/mistral-nemo as reversed results already have 105 rows.
Skipping liquid/lfm-3b as reversed results already have 105 rows.
Skipping meta-llama/llama-3.1-8b-instruct as reversed results already have 105 rows.
Skipping google/gemma-3-4b-it as reversed results already have 105 rows.
Skipping sao10k/l3-lunaris-8b as reversed results already have 105 rows.
skipping nous model
Processing mistralai/mistral-7b-instruct
Comparing sentence 1 with sentence 0 (reversed)
 The text does not provide specific details about a particular climate litigation case, parties involved, court or jurisdiction, statute or regulation, posture, dates, monetary amounts, or sought outcome. It only mentions the potential exposure to climate change-related litigation d

{'mistralai/mistral/nemo':      idx_1  idx_2                                          reasoning   result  \
 0        1      0  The text mentions potential exposure to climat...   text 2   
 1        2      0  **Rationale:** While the text mentions potenti...  neither   
 2        3      0  **Rationale:** The text mentions potential exp...  neither   
 3        4      0  Text 1 provides more specific details about cl...   text 1   
 4        5      0  The text mentions potential exposure to climat...  neither   
 ..     ...    ...                                                ...      ...   
 100     13     11  **Rationale:** Both texts mention climate chan...   text 1   
 101     14     11  Text 1 does not relate to climate litigation, ...   text 2   
 102     13     12  **Rationale:** The text discusses regulatory a...  neither   
 103     14     12  **Rationale:** The text discusses regulatory a...  neither   
 104     14     13  **Rationale:** The text provides some details ...   

In [404]:
#check if all the dfs have 105 rows
for model in pairwise_results_dfs_reversed.keys():
    print(model)
    if len(pairwise_results_dfs_reversed[model]) != 105:
        print(f"Model {model} has {len(pairwise_results_dfs_reversed[model])} rows instead of 105.")


mistralai/mistral/nemo
meta/llama/llama/3.2/3b/instruct
liquid/lfm/7b
meta/llama/llama/3.2/1b/instruct
liquid/lfm/3b
meta-llama/llama-3.2-1b-instruct
liquid/lfm-7b
meta-llama/llama-3.2-3b-instruct
mistralai/mistral-nemo
liquid/lfm-3b
meta-llama/llama-3.1-8b-instruct
google/gemma-3-4b-it
sao10k/l3-lunaris-8b
mistralai/mistral-7b-instruct


In [410]:
# Save both the forward and reversed results to local storage for all models, in separate folders
import os

forward_dir = '../outputs/forward'
reversed_dir = '../outputs/reversed'
os.makedirs(forward_dir, exist_ok=True)
os.makedirs(reversed_dir, exist_ok=True)

for model in pairwise_results_dfs.keys():
    safe_model_name = model.replace("/", "_").replace("-", "_")
    forward_path = os.path.join(forward_dir, f'pairwise_results_dfs_{safe_model_name}_forward.csv')
    reversed_path = os.path.join(reversed_dir, f'pairwise_results_dfs_{safe_model_name}_reversed.csv')
    if model in pairwise_results_dfs:
        pairwise_results_dfs[model].to_csv(forward_path, index=False)
    if model in pairwise_results_dfs_reversed:
        pairwise_results_dfs_reversed[model].to_csv(reversed_path, index=False)


In [406]:
invalid_results = check_invalid_results(pairwise_results_dfs_reversed)
invalid_counts = Counter()
for entry in invalid_results:
    model = entry['model']
    invalid_counts[model] += 1

# Print the percentage of invalid results for each model (out of 105)
for model in pairwise_results_dfs_reversed.keys():
    count = invalid_counts.get(model, 0)
    percent = (count / 105) * 100
    print(f"Model: {model}, Invalid results: {count} ({percent:.2f}%)")

Model: mistralai/mistral/nemo, Invalid results: 0 (0.00%)
Model: meta/llama/llama/3.2/3b/instruct, Invalid results: 0 (0.00%)
Model: liquid/lfm/7b, Invalid results: 0 (0.00%)
Model: meta/llama/llama/3.2/1b/instruct, Invalid results: 0 (0.00%)
Model: liquid/lfm/3b, Invalid results: 0 (0.00%)
Model: meta-llama/llama-3.2-1b-instruct, Invalid results: 18 (17.14%)
Model: liquid/lfm-7b, Invalid results: 32 (30.48%)
Model: meta-llama/llama-3.2-3b-instruct, Invalid results: 18 (17.14%)
Model: mistralai/mistral-nemo, Invalid results: 38 (36.19%)
Model: liquid/lfm-3b, Invalid results: 11 (10.48%)
Model: meta-llama/llama-3.1-8b-instruct, Invalid results: 1 (0.95%)
Model: google/gemma-3-4b-it, Invalid results: 6 (5.71%)
Model: sao10k/l3-lunaris-8b, Invalid results: 0 (0.00%)
Model: mistralai/mistral-7b-instruct, Invalid results: 103 (98.10%)


In [407]:
# Overwrite invalid results in pairwise_results_dfs with rerun slimmer_pairwise results
output_folder = "../outputs/single/"
for entry in invalid_results:
    model = entry['model']
    print(f"Model: {model}")
    idx_1 = entry['idx_1']
    idx_2 = entry['idx_2']
    # Get the model_df used for this model
    model_df = load_full_messages(model, output_folder)
    # Get the texts for idx_1 and idx_2
    text_1 = model_df.loc[model_df.index == idx_1, 'full_message'].values[0]
    text_2 = model_df.loc[model_df.index == idx_2, 'full_message'].values[0]
    # Run slimmer_pairwise on the two texts
    result, final_result = slimmer_pairwise(text_1, text_2)
    # Overwrite the result in the pairwise_results_dfs_reversed
    pairwise_df = pairwise_results_dfs_reversed[model]
    mask = (pairwise_df['idx_1'] == idx_1) & (pairwise_df['idx_2'] == idx_2)
    # Use .at to set the value for the single row
    idxs = pairwise_df.index[mask]
    if len(idxs) == 1:
        pairwise_results_dfs_reversed[model].at[idxs[0], 'result'] = final_result
        pairwise_results_dfs_reversed[model].at[idxs[0], 'reasoning'] = result
    elif len(idxs) == 0:
        print(f"Warning: No matching row found for model={model}, idx_1={idx_1}, idx_2={idx_2}")
    else:
        print(f"Warning: Multiple matching rows found for model={model}, idx_1={idx_1}, idx_2={idx_2}")


Model: meta-llama/llama-3.2-1b-instruct
Text 1 is more specific than Text 2 because it provides more details about the potential impact of climate change-related regulatory activity on the company's business, including specific regulations such as the European Union's Fit for 55 package and the IMO Sulfur Limit. Text 2, on the other hand, provides more general information about the company's concerns regarding climate change regulations and their potential impact on the natural gas industry.

FINAL: text 1
In the following content, extract the final decision. Return ONLY 'text 1' or 'text 2' or 'neither' (no other text). If absent, infer from the last explicit choice.

Text 1 is more specific than Text 2 because it provides more details about the potential impact of climate change-related regulatory activity on the company's business, including specific regulations such as the European Union's Fit for 55 package and the IMO Sulfur Limit. Text 2, on the other hand, provides more general

In [408]:
#check if all the dfs have 105 rows
for model in pairwise_results_dfs_reversed.keys():
    print(model)
    if len(pairwise_results_dfs_reversed[model]) != 105:
        print(f"Model {model} has {len(pairwise_results_dfs_reversed[model])} rows instead of 105.")


mistralai/mistral/nemo
meta/llama/llama/3.2/3b/instruct
liquid/lfm/7b
meta/llama/llama/3.2/1b/instruct
liquid/lfm/3b
meta-llama/llama-3.2-1b-instruct
liquid/lfm-7b
meta-llama/llama-3.2-3b-instruct
mistralai/mistral-nemo
liquid/lfm-3b
meta-llama/llama-3.1-8b-instruct
google/gemma-3-4b-it
sao10k/l3-lunaris-8b
mistralai/mistral-7b-instruct


In [409]:
invalid_results = check_invalid_results(pairwise_results_dfs_reversed)
invalid_counts = Counter()
for entry in invalid_results:
    model = entry['model']
    invalid_counts[model] += 1

# Print the percentage of invalid results for each model (out of 105)
for model in pairwise_results_dfs_reversed.keys():
    count = invalid_counts.get(model, 0)
    percent = (count / 105) * 100
    print(f"Model: {model}, Invalid results: {count} ({percent:.2f}%)")

Model: mistralai/mistral/nemo, Invalid results: 0 (0.00%)
Model: meta/llama/llama/3.2/3b/instruct, Invalid results: 0 (0.00%)
Model: liquid/lfm/7b, Invalid results: 0 (0.00%)
Model: meta/llama/llama/3.2/1b/instruct, Invalid results: 0 (0.00%)
Model: liquid/lfm/3b, Invalid results: 0 (0.00%)
Model: meta-llama/llama-3.2-1b-instruct, Invalid results: 0 (0.00%)
Model: liquid/lfm-7b, Invalid results: 0 (0.00%)
Model: meta-llama/llama-3.2-3b-instruct, Invalid results: 0 (0.00%)
Model: mistralai/mistral-nemo, Invalid results: 0 (0.00%)
Model: liquid/lfm-3b, Invalid results: 0 (0.00%)
Model: meta-llama/llama-3.1-8b-instruct, Invalid results: 0 (0.00%)
Model: google/gemma-3-4b-it, Invalid results: 0 (0.00%)
Model: sao10k/l3-lunaris-8b, Invalid results: 0 (0.00%)
Model: mistralai/mistral-7b-instruct, Invalid results: 0 (0.00%)


In [411]:
for model in pairwise_results_dfs_reversed:
    pairwise_results_dfs_reversed[model] = convert_result_to_index(pairwise_results_dfs_reversed[model])

In [412]:
# Count the percentage of times each original model predicts "text 1", "text 2", or "neither"
model_percentages = {}
for model, df in pairwise_results_dfs_reversed.items():
    # Normalize result column to strip whitespace and lowercase
    results = df['result'].astype(str).str.strip().str.lower()
    total = len(results)
    text1_count = ((results == 'text 1') | (results == 'text 1\n')).sum()
    text2_count = ((results == 'text 2') | (results == 'text 2\n')).sum()
    neither_count = total - text1_count - text2_count
    model_percentages[model] = {
        'text 1': round(100 * text1_count / total, 2) if total > 0 else 0,
        'text 2': round(100 * text2_count / total, 2) if total > 0 else 0,
        'neither': round(100 * neither_count / total, 2) if total > 0 else 0
    }

model_counts_df = pd.DataFrame.from_dict(model_percentages, orient='index').reset_index().rename(columns={'index': 'Model'})


In [413]:
model_counts_df

,Model,text 1,text 2,neither
0,mistralai/mistral/nemo,25.71,26.67,47.62
1,meta/llama/llama/3.2/3b/instruct,32.38,54.29,13.33
2,liquid/lfm/7b,84.76,12.38,2.86
3,meta/llama/llama/3.2/1b/instruct,44.76,29.52,25.71
4,liquid/lfm/3b,22.86,8.57,68.57
5,meta-llama/llama-3.2-1b-instruct,31.43,45.71,22.86
6,liquid/lfm-7b,88.57,6.67,4.76
7,meta-llama/llama-3.2-3b-instruct,37.14,56.19,6.67
8,mistralai/mistral-nemo,23.81,19.05,57.14
9,liquid/lfm-3b,22.86,10.48,66.67


In [414]:
# Compare forward and reversed results per model: compute, for each model, the percentage of pairs where the forward and reversed winner_idx are the same for that model only

percent_same_by_model = {}

for model in pairwise_results_dfs.keys():
    if model not in pairwise_results_dfs or model not in pairwise_results_dfs_reversed:
        continue  # Only process if present in both sets

    df_forward = pairwise_results_dfs[model]
    df_reversed = pairwise_results_dfs_reversed[model]

    # To compare, align each pair: (idx_1, idx_2) forward with (idx_2, idx_1) reversed
    merged = pd.merge(
        df_forward,
        df_reversed,
        left_on=['idx_1', 'idx_2'],
        right_on=['idx_2', 'idx_1'],
        suffixes=('_forward', '_reversed')
    )

    # Compare winner_idx for each model only
    if 'winner_idx_forward' in merged.columns and 'winner_idx_reversed' in merged.columns:
        model_total = merged.shape[0]
        model_same_winner = (merged['winner_idx_forward'] == merged['winner_idx_reversed']).sum()
        percent_same = (model_same_winner / model_total * 100) if model_total > 0 else 0.0
        percent_same_by_model[model] = percent_same
        print(f"[{model}] Forward vs. reversed winner_idx SAME for {model_same_winner}/{model_total} ({percent_same:.2f}%)")
    else:
        print(f"[{model}] Columns for winner_idx not found, skipping.")

# The dictionary percent_same_by_model can be accessed for further per-model comparison or plotting.


[meta-llama/llama-3.2-1b-instruct] Forward vs. reversed winner_idx SAME for 42/105 (40.00%)
[liquid/lfm-7b] Forward vs. reversed winner_idx SAME for 84/105 (80.00%)
[meta-llama/llama-3.2-3b-instruct] Forward vs. reversed winner_idx SAME for 39/105 (37.14%)
[mistralai/mistral-nemo] Forward vs. reversed winner_idx SAME for 23/105 (21.90%)
[liquid/lfm-3b] Forward vs. reversed winner_idx SAME for 14/105 (13.33%)
[meta-llama/llama-3.1-8b-instruct] Forward vs. reversed winner_idx SAME for 36/105 (34.29%)
[google/gemma-3-4b-it] Forward vs. reversed winner_idx SAME for 4/105 (3.81%)
[sao10k/l3-lunaris-8b] Forward vs. reversed winner_idx SAME for 26/105 (24.76%)
[mistralai/mistral-7b-instruct] Forward vs. reversed winner_idx SAME for 65/105 (61.90%)


_____ 

Ok, so for now, it looks like liquidlfm is the best but this has gotten messy. I am going to restart, iterating the prompt a little bit, and doing forward and reverse in the same run, in another notebook called pairwise_comparisons.ipynb